# HDBSCAN on TikZ code embeddings

Clusters TikZ samples from the TikZ/LaTeX source code instead of image embeddings.

Tests CodeBERT, GraphCodeBERT and UniXcoder, plus two combined embeddings, with PCA and UMAP followed by HDBSCAN.

In [5]:
from pathlib import Path
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import concatenate_datasets, load_dataset
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm
from umap import UMAP

In [6]:
DATIKZ_DIR = Path("/home/jonas/Datasets/TikZ/DatikZ-v4")
BENCHMARK_DIR = Path("/home/jonas/Datasets/TikZ/benchmark-not-clean")
DATASETS_TO_USE = ("datikz", "benchmark")

DATIKZ_IMAGE_COL = "png_image"
DATIKZ_CODE_COL = "tikz_code"
DATIKZ_ID_COL = "file_id"
DATIKZ_SOURCE_COL = "source"

BENCHMARK_IMAGE_COL = "image"
BENCHMARK_CODE_COL = "code"
BENCHMARK_ID_COL = "uri"
BENCHMARK_SOURCE_COL = "origin"

N_SAMPLES_DATIKZ = None
N_SAMPLES_BENCHMARK = None

MODEL_IDS = {
    "codebert": "microsoft/codebert-base",
    "graphcodebert": "microsoft/graphcodebert-base",
    "unixcoder": "microsoft/unixcoder-base",
}

EMBEDDING_VARIANTS = [
    "codebert",
    "graphcodebert",
    "unixcoder",
    "codebert+unixcoder",
    "graphcodebert+unixcoder",
]

MAX_TOKENS = 512
BATCH_SIZE = 32
SEED = 42

REDUCERS = ["pca", "umap"]
COMPONENTS_TO_TEST = [
    5,
    10,
    15,
    20,
    25,
    30,
    35,
    40,
    80,
    128,
]

UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.0
UMAP_METRIC = "cosine"

MIN_CLUSTER_SIZE = 200
MIN_SAMPLES = 20
SILHOUETTE_SAMPLE_SIZE = 5_000

AUTO_SELECT_BEST_RUN = False
FINAL_EMBEDDING_VARIANT = "codebert"
FINAL_REDUCER = "umap"
FINAL_COMPONENTS = 80

OUTPUT_PREFIX = "tikz_code_hdbscan"

In [7]:
def load_local(path):
    files = sorted(path.rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No Parquet files found in {path}")
    return load_dataset("parquet", data_files=[str(p) for p in files], split="train")


def prepare(ds, name, image_col, code_col, id_col, source_col, n, seed):
    required = {image_col, code_col, id_col, source_col}
    missing = required - set(ds.column_names)
    if missing:
        raise ValueError(f"{name} is missing columns: {sorted(missing)}")

    if image_col != "image":
        ds = ds.rename_column(image_col, "image")
    if code_col != "code":
        ds = ds.rename_column(code_col, "code")

    ds = ds.map(
        lambda row, i: {
            "dataset": name,
            "sample_id": f"{name}:{row[id_col] if row[id_col] is not None else i}",
            "source": row[source_col],
        },
        with_indices=True,
    )
    ds = ds.select_columns(["image", "code", "dataset", "sample_id", "source"])
    if n is not None:
        ds = ds.shuffle(seed=seed).select(range(min(n, len(ds))))
    return ds


datasets = []
if "datikz" in DATASETS_TO_USE:
    datasets.append(prepare(load_local(DATIKZ_DIR), "datikz", DATIKZ_IMAGE_COL, DATIKZ_CODE_COL, DATIKZ_ID_COL, DATIKZ_SOURCE_COL, N_SAMPLES_DATIKZ, SEED))
if "benchmark" in DATASETS_TO_USE:
    datasets.append(prepare(load_local(BENCHMARK_DIR), "benchmark", BENCHMARK_IMAGE_COL, BENCHMARK_CODE_COL, BENCHMARK_ID_COL, BENCHMARK_SOURCE_COL, N_SAMPLES_BENCHMARK, SEED + 1))

sample = concatenate_datasets(datasets).shuffle(seed=SEED)
print(sample)
print(pd.Series(sample["dataset"]).value_counts())

Dataset({
    features: ['image', 'code', 'dataset', 'sample_id', 'source'],
    num_rows: 595447
})
datikz       427753
benchmark    167694
Name: count, dtype: int64


## Generate TikZ code embeddings

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)


def encode(name, model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id).eval().to(device)
    chunks = []

    for start in tqdm(range(0, len(sample), BATCH_SIZE), desc=name):
        codes = [str(x or "") for x in sample[start:start + BATCH_SIZE]["code"]]
        inputs = tokenizer(
            codes,
            padding=True,
            truncation=True,
            max_length=MAX_TOKENS,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            out = model(**inputs)
            z = mean_pool(out.last_hidden_state, inputs["attention_mask"])
            z = F.normalize(z.float(), dim=-1)

        chunks.append(z.cpu().numpy())

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.concatenate(chunks)


base_embeddings = {
    name: encode(name, model_id)
    for name, model_id in MODEL_IDS.items()
}

Device: cuda


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 69063.68it/s]
[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
unixcoder: 100%|██████████| 18608/18608 [46:43<00:00,  6.64it/s] 


In [ ]:
def norm(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)


def combine(a, b):
    return norm(np.concatenate([norm(a), norm(b)], axis=1))


embeddings = {
    "codebert": base_embeddings["codebert"],
    "graphcodebert": base_embeddings["graphcodebert"],
    "unixcoder": base_embeddings["unixcoder"],
    "codebert+unixcoder": combine(base_embeddings["codebert"], base_embeddings["unixcoder"]),
    "graphcodebert+unixcoder": combine(base_embeddings["graphcodebert"], base_embeddings["unixcoder"]),
}

for name, E in embeddings.items():
    print(name, E.shape)

## PCA / UMAP / HDBSCAN sweep

In [ ]:
def evaluate(X):
    model = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, n_jobs=-1)
    labels = model.fit_predict(X)
    mask = labels >= 0
    clusters = np.unique(labels[mask])
    score = np.nan

    if len(clusters) >= 2:
        score = silhouette_score(
            X[mask], labels[mask],
            sample_size=min(SILHOUETTE_SAMPLE_SIZE, mask.sum()),
            random_state=SEED,
        )

    return {
        "clusters": len(clusters),
        "noise": float(np.mean(labels == -1)),
        "silhouette": score,
    }


rows = []

for embedding_name in EMBEDDING_VARIANTS:
    E = embeddings[embedding_name]

    if "pca" in REDUCERS:
        max_pc = min(max(COMPONENTS_TO_TEST), len(E) - 1, E.shape[1])
        pca = PCA(n_components=max_pc, random_state=SEED)
        X_all = pca.fit_transform(E)

        for pc in tqdm(COMPONENTS_TO_TEST, desc=f"{embedding_name} PCA"):
            pc = min(pc, max_pc)
            rows.append({
                "embedding": embedding_name,
                "reducer": "pca",
                "components": pc,
                "variance": pca.explained_variance_ratio_[:pc].sum(),
                **evaluate(X_all[:, :pc]),
            })

    if "umap" in REDUCERS:
        for pc in tqdm(COMPONENTS_TO_TEST, desc=f"{embedding_name} UMAP"):
            reducer = UMAP(
                n_components=pc,
                n_neighbors=UMAP_N_NEIGHBORS,
                min_dist=UMAP_MIN_DIST,
                metric=UMAP_METRIC,
                random_state=SEED,
            )
            X = reducer.fit_transform(E)
            rows.append({
                "embedding": embedding_name,
                "reducer": "umap",
                "components": pc,
                "variance": np.nan,
                **evaluate(X),
            })

results = pd.DataFrame(rows)
ranking = results.dropna(subset=["silhouette"]).sort_values(["silhouette", "noise"], ascending=[False, True])
display(ranking)

In [ ]:
for metric in ["silhouette", "noise", "clusters"]:
    plt.figure(figsize=(11, 5))
    for embedding_name in EMBEDDING_VARIANTS:
        for reducer_name in REDUCERS:
            part = results[(results.embedding == embedding_name) & (results.reducer == reducer_name)]
            plt.plot(part.components, part[metric], marker="o", label=f"{embedding_name} + {reducer_name}")
    plt.xlabel("Dimensions")
    plt.ylabel(metric)
    plt.legend(fontsize=8)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## Fit selected run

In [ ]:
if AUTO_SELECT_BEST_RUN:
    best = ranking.iloc[0]
    selected_embedding = str(best.embedding)
    selected_reducer = str(best.reducer)
    selected_components = int(best.components)
else:
    selected_embedding = FINAL_EMBEDDING_VARIANT
    selected_reducer = FINAL_REDUCER
    selected_components = FINAL_COMPONENTS

E = embeddings[selected_embedding]

if selected_reducer == "pca":
    reducer = PCA(n_components=selected_components, random_state=SEED)
else:
    reducer = UMAP(
        n_components=selected_components,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=SEED,
    )

X = reducer.fit_transform(E)
clusterer = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, n_jobs=-1)
labels = clusterer.fit_predict(X)
probabilities = clusterer.probabilities_

mask = labels >= 0
score = silhouette_score(
    X[mask], labels[mask],
    sample_size=min(SILHOUETTE_SAMPLE_SIZE, mask.sum()),
    random_state=SEED,
)

print("Embedding:", selected_embedding)
print("Reducer:", selected_reducer)
print("Dimensions:", selected_components)
print("Clusters:", len(set(labels)) - (1 if -1 in labels else 0))
print("Noise:", f"{np.mean(labels == -1):.2%}")
print("Silhouette:", round(score, 4))

## Inspect clusters

In [ ]:
def show_cluster(cluster_id, n=20, cols=5):
    idx = np.where(labels == cluster_id)[0]
    if cluster_id >= 0:
        idx = idx[np.argsort(probabilities[idx])[::-1]]
    idx = idx[:n]
    rows = int(np.ceil(len(idx) / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")

    for ax, i in zip(axes.ravel(), idx):
        row = sample[int(i)]
        ax.imshow(row["image"])
        ax.set_title(
            f"{row['dataset']} | p={probabilities[i]:.2f}" if cluster_id >= 0 else row["dataset"],
            fontsize=8,
        )

    fig.suptitle("noise" if cluster_id == -1 else f"cluster {cluster_id}")
    plt.tight_layout()
    plt.show()


for cluster_id in sorted(x for x in set(labels) if x >= 0):
    show_cluster(cluster_id)

if -1 in labels:
    show_cluster(-1)

## Export assignments

In [ ]:
assignments = pd.DataFrame({
    "sample_index": np.arange(len(sample)),
    "dataset": sample["dataset"],
    "sample_id": sample["sample_id"],
    "source": sample["source"],
    "cluster": labels,
    "membership_probability": probabilities,
})

assignments["class_name"] = np.where(
    assignments.cluster == -1,
    "noise",
    "class_" + (assignments.cluster + 1).astype(str),
)

output_file = f"{OUTPUT_PREFIX}_{selected_embedding}_{selected_reducer}{selected_components}.csv"
assignments.to_csv(output_file, index=False)
display(assignments.head())
print("Saved:", output_file)